In [1]:
from DeviceClasses.E5063A_VNA_Class import VNA
from DeviceClasses.RS_FSW_new import FSW
from DeviceClasses.SMA100B import Signal_Generator
from DeviceClasses.Utilities.Fitting import fit_S21_peak, S21_eq, linear_voltage_to_dB, dBm_to_watts, watts_to_dBm, RMS_to_dBm, dBm_to_RMS, RMS_squared_to_dBm
from DeviceClasses.Utilities.data_management import read_file
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import os
%matplotlib qt



In [2]:


base_path = r"D:\Bulk_He_2_1\data\2025_01_20_S21_vs_power_with_FSW_fast_long\\"
S21_data_path = base_path


In [3]:
#organisation of dictionary
# SA_data = {"10[dBm]":{
#     "average_21":{
#         "freq"          :{},
#         "time"          :{},
#         "IQ_voltage_mag":{},
#     }
# }}
S21_data = {}

#load in file
for file in os.listdir(S21_data_path):
    path =   S21_data_path + file
    #print(f"file:{file}")
    if ".tsv" not in path: continue

    data_pandas_temp, meta_data_temp = read_file(path)
    print(f"meta_data:{meta_data_temp.keys()}")
    power = float(meta_data_temp["power"])
    
    print(f"power:{power}")


    #skip data that was taken when refilling fridge which was too noisy (noisy)
    try: 
        fridge_refill = meta_data_temp["fridge_refill"]
        try:
            fridge_refill_trial = meta_data_temp["fridge_refill_trial"]
        except KeyError:
            continue
        #print(f"fridge_refill_trial:{type(fridge_refill_trial)}")
        if fridge_refill_trial not in ['False']: continue

    except KeyError: pass
    
    if not power in S21_data.keys():
        S21_data[power] = {}
    if not file in S21_data[power].keys():
        S21_data[power][file] = {}

    S21_data[power][file] = {"data":data_pandas_temp,
                                    "meta_data":meta_data_temp}
    print(f"saved")

print(f"data:{S21_data.keys()}")    

meta_data:dict_keys(['cavity_pressure', 'temperature', 'fridge_refill', 'fridge_refill_trial', 'daytime', 'span', 'sweep_points', 'power', 'band_width', 'extra_meta', 'save_location'])
power:-15.0
saved
meta_data:dict_keys(['cavity_pressure', 'temperature', 'fridge_refill', 'fridge_refill_trial', 'daytime', 'span', 'sweep_points', 'power', 'band_width', 'extra_meta', 'save_location'])
power:-14.0
saved
meta_data:dict_keys(['cavity_pressure', 'temperature', 'fridge_refill', 'fridge_refill_trial', 'daytime', 'span', 'sweep_points', 'power', 'band_width', 'extra_meta', 'save_location'])
power:-13.0
saved
meta_data:dict_keys(['cavity_pressure', 'temperature', 'fridge_refill', 'fridge_refill_trial', 'daytime', 'span', 'sweep_points', 'power', 'band_width', 'extra_meta', 'save_location'])
power:-12.0
saved
meta_data:dict_keys(['cavity_pressure', 'temperature', 'fridge_refill', 'fridge_refill_trial', 'daytime', 'span', 'sweep_points', 'power', 'band_width', 'extra_meta', 'save_location'])
pow

In [4]:
def save_data_3(data, file_name, file_location = "", headers = None, meta_data = {}, delimiter = "\t"):
    
    data_shape = np.shape(data)
    print(f"datashape:{data_shape}")
    with open(file_location+file_name, "w") as file1:
        for key, value in meta_data.items():
            file1.write(f"#\t{key}\t{value}\n")
        if headers != None:
            file1.write(headers+"\n")
        
        try:
            for  i in range(data_shape[1]):
                for l in range(data_shape[0]):
                    file1.write(f"{data[l,i]}\t")
                file1.write("\n")
        except IndexError as e:
            print(f"index error:\n{e}\ndata:\n{data}")
            for  i in range(data_shape[0]):
                file1.write(f"{data[i]}\t")
                file1.write("\n")


#https://knowledge.ni.com/KnowledgeArticleDetails?id=kA03q000000ow2XCAQ&l=en-GBs

def Vpp_to_Vrms(V_pp):
    Vrms = V_pp/(2*np.sqrt(2))
    return Vrms

def Vp_to_Vrms(V_pp):
    Vrms = V_pp/(np.sqrt(2))
    return Vrms

def Vrms_to_Vpp(Vrms):
    V_pp = Vrms*(2*np.sqrt(2))
    return V_pp

def Vrms_to_Vp(Vrms):
    V_p = Vrms*(np.sqrt(2))
    return V_p

def Vrms_to_watts(Vrms, Z_0 = 50):
    return (Vrms**2)/Z_0

def watts_to_dBw(watts):
    return 10*np.log10(watts)

def RMS_to_dBw(Vrms, Z_0 = 50):
    watts = Vrms_to_watts(Vrms, Z_0)
    return watts_to_dBw(watts)

def RMS_to_dBm(Vrms, Z_0 = 50):
    return RMS_to_dBw(Vrms, Z_0) + 30

def dBw_to_watts(dBw):
    return 10**(dBw/10)

def dBm_to_dBw(dBm):
    return dBm - 30

def dBm_to_RMS(dBm, Z_0 = 50):
    dBw = dBm_to_dBw(dBm)
    watts = dBw_to_watts(dBw)
    Vrms = np.sqrt(watts * Z_0)
    return Vrms

def watts_to_dBm(watts):
    dBw = watts_to_dBw(watts)
    return dBw + 30

def dBm_to_watts(dBm):
    dBw = dBm_to_dBw(dBm)
    watts = dBw_to_watts(dBw)
    return watts

def Vrms_squared_to_watts(Vrms_squared,Z_0 = 50):
    watts = Vrms_squared/Z_0
    return watts





# def RMS_to_dBm(V, Z_0 = 50):
#     return 20*np.log10((V)/(np.sqrt(Z_0))) +30

# def RMS_squared_to_dBm(V_square, Z_0 = 50):
#     return 10*np.log10((V_square)/(0.001*Z_0))

# def dBm_to_RMS(dBm, Z_0 = 50):
#     return np.sqrt((Z_0*0.001)*10**(dBm/10))

In [5]:
print(f"data:{S21_data.keys()}")
#SA_data["10[dBm]"].pop("average_0")


data:dict_keys([-15.0, -14.0, -13.0, -12.0, -11.0, -10.0, -9.0, -8.0, -7.0, -6.0, -5.0, -4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0])


In [6]:
plt.close()

fig_S21, ax_S21 = plt.subplots()
ax_S21.set(title = f"S21 data and Fit", ylabel = "|S21| [dB]", xlabel = "frequency[Hz]")

fig_S21_popt, (ax_f0, ax_k_tot, ax_k_ext) = plt.subplots(3, sharex = True)
ax_f0.set(title = f"S21 fit vs probe_power", ylabel = "f_0[Hz]", )
ax_k_tot.set( ylabel = "k_tot[Hz]",)
ax_k_ext.set(ylabel = "k_ext[Hz]", xlabel = "probe_power[dBm]")
fig_S21_popt.tight_layout()

magnitudes_averaged = []
f_0_list_averaged = []
k_tot_list_averaged =[]
k_ext_list_averaged = []
probe_power_averaged = []

freq_offset = 0
for power, power_dict in S21_data.items():
    magnitudes_temp = []
    f_0_list_temp = []
    k_tot_list_temp =[]
    k_ext_list_temp = []
    probe_power_temp = []
    for file_name, file_name_dict in S21_data[power].items():
        print(f"power:{power}")
        print(f"power_dict:{file_name_dict['data'].keys()}")
        data_pandas_temp = file_name_dict["data"]
        meta_data = file_name_dict["meta_data"]

        band_width = meta_data["band_width"]

        mag = data_pandas_temp["S21[dB]"]
        freq = data_pandas_temp["frequency[Hz]"]

        signal_applied = [band_width, 0]
        popt, pcov, guesses, amplitude_linear_voltage, frequencies = fit_S21_peak(mag, freq, guesses = False, signal_applied = signal_applied)
        
        f_0, k_tot, k_ext, mx, c = popt
        f_0_list_temp.append(f_0)
        k_tot_list_temp.append(k_tot)
        k_ext_list_temp.append(k_ext)
        probe_power_temp.append(power)
        magnitudes_temp.append(mag)

    magnitudes_temp = np.mean(magnitudes_temp,axis = 0)
    f_0_list_temp  = np.mean(f_0_list_temp,axis = 0)
    k_tot_list_temp  = np.mean(k_tot_list_temp,axis = 0)
    k_ext_list_temp  = np.mean(k_ext_list_temp,axis = 0)
    probe_power_temp = np.mean(probe_power_temp,axis = 0)

    f_0_list_averaged.append(f_0_list_temp)
    k_tot_list_averaged.append(k_tot_list_temp)
    k_ext_list_averaged.append(k_ext_list_temp)
    probe_power_averaged.append(probe_power_temp)
    magnitudes_averaged.append(magnitudes_temp)
    
        


#Plot the data
    if float(power) in [-15,-10,-5,0,5,10,15]:
        print(f"plotting for:{power}")
        freq_offset = float(power)*1e5
        ax_S21.plot(freq + freq_offset, mag, '-', label=f'Data (S21 in Power), probe pwr:{power}')
        ax_S21.plot(freq + freq_offset, linear_voltage_to_dB(S21_eq(freq, *popt)), '--', label=f'Fit probe pwr:{power}, popt:{popt}')
        fig_S21.show()
        ax_S21.legend(loc = "upper right")
        

        

probe_power_averaged= np.array(probe_power_averaged)
sorted_indices = np.argsort(probe_power_averaged)
probe_power_averaged = probe_power_averaged[sorted_indices]

f_0_list_averaged = np.array(f_0_list_averaged)[sorted_indices]
k_tot_list_averaged = np.array(k_tot_list_averaged)[sorted_indices]
k_ext_list_averaged = np.array(k_ext_list_averaged)[sorted_indices]


line_f0, = ax_f0.plot(probe_power_averaged, f_0_list_averaged, "-x")
line_k_tot, = ax_k_tot.plot(probe_power_averaged, k_tot_list_averaged, "-x")
line_k_ext, = ax_k_ext.plot(probe_power_averaged, k_ext_list_averaged, "-x")
fig_S21_popt.show()


print(probe_power_averaged)




power:-15.0
power_dict:Index(['frequency[Hz]', 'S21[dB]', 'S21_sig[dB]'], dtype='object')
frequencies:[6.18814649e+09 6.18816649e+09 6.18818649e+09 6.18820649e+09
 6.18822649e+09 6.18824649e+09 6.18826649e+09 6.18828649e+09
 6.18830649e+09 6.18832649e+09 6.18834649e+09 6.18836649e+09
 6.18838649e+09 6.18840649e+09 6.18842649e+09 6.18844649e+09
 6.18846649e+09 6.18848649e+09 6.18850649e+09 6.18852649e+09
 6.18854649e+09 6.18856649e+09 6.18858649e+09 6.18860649e+09
 6.18862649e+09 6.18864649e+09 6.18866649e+09 6.18868649e+09
 6.18870649e+09 6.18872649e+09 6.18874649e+09 6.18876649e+09
 6.18878649e+09 6.18880649e+09 6.18882649e+09 6.18884649e+09
 6.18886649e+09 6.18888649e+09 6.18890649e+09 6.18892649e+09
 6.18894649e+09 6.18896649e+09 6.18898649e+09 6.18900649e+09
 6.18902649e+09 6.18904649e+09 6.18906649e+09 6.18908649e+09
 6.18910649e+09 6.18912649e+09 6.18914649e+09]
plotting for:-15.0
power:-14.0
power_dict:Index(['frequency[Hz]', 'S21[dB]', 'S21_sig[dB]'], dtype='object')
frequencie

In [7]:
31250.0/0.016016

1951173.8261738264